In [1]:
#%% Imports & Setup
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.io import loadmat
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import lr_scheduler
from torchinfo import summary
from time import time
import h5py
import rasterio
from tqdm import tqdm
from einops import rearrange

#%% Preprocessing Functions (from Image_Preproc.py)
def ScaleData(X, min_value, max_value):
    """
    Scales the data X between 0 and 1.
    """
    min_value = np.float16(min_value)
    max_value = np.float16(max_value)
    X -= min_value
    X /= (max_value - min_value)
    return X

def extract_overlapping_patches(image, patch_size, stride):
    """
    Extract overlapping patches from an image.
    
    Args:
        image (numpy array or torch.Tensor): Input image of shape [C, H, W] (or [H, W] for single channel).
        patch_size (int): Size of each patch.
        stride (int): Stride for extracting patches (controls overlap).
    
    Returns:
        patches (numpy array): Extracted patches.
    """
    image = torch.tensor(image)
    if len(image.shape) > 2:
        patches = image.unfold(1, patch_size, stride).unfold(2, patch_size, stride)
        patches = rearrange(patches, "c h w p1 p2 -> (h w) c p1 p2")
    else:
        patches = image.unfold(0, patch_size, stride).unfold(1, patch_size, stride)
        patches = rearrange(patches, "h w p1 p2 -> (h w) p1 p2")
    return patches.numpy()


In [6]:
#%% Set Working Directory & Load Data
from scipy.io import loadmat


def load_mat_data(file_path):
    try:
        data = h5py.File(file_path, 'r')
        hsi_data = np.array(data['HSI'])  # Hyperspectral image data
        gt_data = np.array(data['GT'])    # Ground truth data
        print(f"Loaded file: {file_path}")
        print(f"HSI shape: {hsi_data.shape}, GT shape: {gt_data.shape}")
        return hsi_data, gt_data
    except Exception as e:
        print(f"Error loading .mat file: {e}")
        return None, None

file_path = r'C:\Users\ChloeAtherton\Capstone\data\NC12.mat'
data, gt = load_mat_data(file_path)
if data is None or gt is None:
    raise ValueError("Failed to load data from the specified file.")

data = np.float16(data)

# Process ground truth: set 0 values to 255 and subtract 1 from remaining labels
gt[gt == 0] = 255
gt[gt != 255] -= 1

min_value = np.min(data)
max_value = np.max(data)
data = ScaleData(data, min_value, max_value)

#%% Patchify Data
h, w = data.shape[1], data.shape[2]
patch_size = 32
pad_h = patch_size * (h // patch_size + 1) - h  # Padding needed for height
pad_w = patch_size * (w // patch_size + 1) - w  # Padding needed for width

data = np.pad(data, ((0, 0), (0, pad_h), (0, pad_w)), mode='constant')
gt = np.pad(gt, ((0, pad_h), (0, pad_w)), mode='constant', constant_values=255)

# Use overlapping patches
stride = 8
data = extract_overlapping_patches(data, patch_size, stride)
gt = extract_overlapping_patches(gt, patch_size, stride)

# Filter patches: keep those with at least 30% labeled pixels (pixels with value < 100)
filt = np.sum(gt < 100, axis=(1,2))
mask = filt > ((patch_size**2) * 0.3)
data = data[mask]
gt = gt[mask]

print(f"Dimension of patchified data: {data.shape}")
print(f"Dimension of patchified GT: {gt.shape}")

Loaded file: C:\Users\ChloeAtherton\Capstone\data\NC12.mat
HSI shape: (270, 2884, 682), GT shape: (2884, 682)
Dimension of patchified data: (8678, 270, 32, 32)
Dimension of patchified GT: (8678, 32, 32)


In [5]:
#%% Train-Validation Split
train_idx, val_idx = train_test_split(np.arange(data.shape[0]), test_size=0.2, random_state=42)
train_data = data[train_idx]
train_gt = gt[train_idx]
validation_data = data[val_idx]
validation_gt = gt[val_idx]

print(f"Train data shape: {train_data.shape}")
print(f"Train GT shape: {train_gt.shape}")
print(f"Validation data shape: {validation_data.shape}")
print(f"Validation GT shape: {validation_gt.shape}")


Train data shape: (6942, 270, 32, 32)
Train GT shape: (6942, 32, 32)
Validation data shape: (1736, 270, 32, 32)
Validation GT shape: (1736, 32, 32)


In [4]:
#%% Define the Segmentation Model
class SimpleSegmentationModel(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, init_features=32):
        super(SimpleSegmentationModel, self).__init__()
        features = init_features
        
        # Encoder
        self.encoder1 = self._block(in_channels, features)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder2 = self._block(features, features * 2)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Bottleneck
        self.bottleneck = self._block(features * 2, features * 4)
        
        # Decoder
        self.upconv2 = nn.ConvTranspose2d(features * 4, features * 2, kernel_size=2, stride=2)
        self.decoder2 = self._block(features * 4, features * 2)
        self.upconv1 = nn.ConvTranspose2d(features * 2, features, kernel_size=2, stride=2)
        self.decoder1 = self._block(features * 2, features)
        
        # Final layer
        self.conv = nn.Conv2d(features, out_channels, kernel_size=1)
    
    def forward(self, x):
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(self.pool1(enc1))
        bottleneck = self.bottleneck(self.pool2(enc2))
        dec2 = self.upconv2(bottleneck)
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.decoder2(dec2)
        dec1 = self.upconv1(dec2)
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.decoder1(dec1)
        return self.conv(dec1)
    
    def _block(self, in_channels, features):
        return nn.Sequential(
            nn.Conv2d(in_channels, features, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(features),
            nn.ReLU(inplace=True),
            nn.Conv2d(features, features, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(features),
            nn.ReLU(inplace=True)
        )

In [7]:
#%% Prepare Data for Training
batch_size = 128
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Convert NumPy arrays to PyTorch tensors
train_data = torch.tensor(train_data, dtype=torch.float32).to(device)
train_gt = torch.tensor(train_gt, dtype=torch.int64).to(device)
validation_data = torch.tensor(validation_data, dtype=torch.float32).to(device)
validation_gt = torch.tensor(validation_gt, dtype=torch.int64).to(device)

train_dataset = TensorDataset(train_data, train_gt)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataset = TensorDataset(validation_data, validation_gt)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

#%% Model Definition & Setup
input_shape = train_data[0].shape  # shape of a single patch
nb_classes = 13  # number of classes
model = SimpleSegmentationModel(in_channels=input_shape[0], out_channels=nb_classes).to(device)
print(summary(model, (1,) + input_shape))

criterion = nn.CrossEntropyLoss(ignore_index=255)
optimizer = optim.Adam(model.parameters(), weight_decay=1e-4)
scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

Layer (type:depth-idx)                   Output Shape              Param #
SimpleSegmentationModel                  [1, 13, 32, 32]           --
├─Sequential: 1-1                        [1, 32, 32, 32]           --
│    └─Conv2d: 2-1                       [1, 32, 32, 32]           77,760
│    └─BatchNorm2d: 2-2                  [1, 32, 32, 32]           64
│    └─ReLU: 2-3                         [1, 32, 32, 32]           --
│    └─Conv2d: 2-4                       [1, 32, 32, 32]           9,216
│    └─BatchNorm2d: 2-5                  [1, 32, 32, 32]           64
│    └─ReLU: 2-6                         [1, 32, 32, 32]           --
├─MaxPool2d: 1-2                         [1, 32, 16, 16]           --
├─Sequential: 1-3                        [1, 64, 16, 16]           --
│    └─Conv2d: 2-7                       [1, 64, 16, 16]           18,432
│    └─BatchNorm2d: 2-8                  [1, 64, 16, 16]           128
│    └─ReLU: 2-9                         [1, 64, 16, 16]           --
│  

In [2]:
#%% Define Metrics
def calculate_accuracy(output, target, ignore_index=255):
    preds = torch.argmax(output, dim=1)
    mask = (target != ignore_index) if ignore_index is not None else torch.ones_like(target, dtype=torch.bool)
    correct = (preds[mask] == target[mask]).sum().item()
    total = mask.sum().item()
    return correct / (total + 1e-6)

def calculate_iou(output, target, num_classes, ignore_index=255):
    preds = torch.argmax(output, dim=1)
    intersection = torch.zeros(num_classes)
    union = torch.zeros(num_classes)
    for cls in range(num_classes):
        pred_mask = (preds == cls)
        target_mask = (target == cls)
        if ignore_index is not None:
            pred_mask = pred_mask & (target != ignore_index)
            target_mask = target_mask & (target != ignore_index)
        intersection[cls] = (pred_mask & target_mask).sum().item()
        union[cls] = (pred_mask | target_mask).sum().item()
    return intersection / (union + 1e-6)



In [3]:
import torch
print(f"Is CUDA available: {torch.cuda.is_available()}")
print(f"PyTorch version: {torch.__version__}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Is CUDA available: True
PyTorch version: 2.5.1+cu124
Using device: cuda


In [16]:
#%% Training Loop (with GPU support)
nb_epoch = 100
best_val_loss = float('inf')

for epoch in range(nb_epoch):
    start_time = time()
    model.train()
    train_loss = 0.0
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    scheduler.step()
    
    # Validation
    model.eval()
    val_loss = 0.0
    total_accuracy = 0.0
    total_iou = torch.zeros(nb_classes).to(device)
    total_samples = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            total_iou += calculate_iou(outputs, labels, nb_classes, ignore_index=255).to(device)
            total_accuracy += calculate_accuracy(outputs, labels, ignore_index=255)
            total_samples += 1
    val_loss /= len(val_loader)
    mean_iou = (total_iou / total_samples).mean().item()
    mean_accuracy = total_accuracy / total_samples
    epoch_time = time() - start_time
    print(f"Epoch [{epoch+1}/{nb_epoch}], Val Loss: {val_loss:.4f}, Accuracy: {mean_accuracy:.4f}, IOU: {mean_iou:.4f}, Time: {epoch_time:.2f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pth')


Epoch [1/100], Val Loss: 0.4946, Accuracy: 0.8885, IOU: 0.3404, Time: 38.18
Epoch [2/100], Val Loss: 0.2636, Accuracy: 0.9371, IOU: 0.3919, Time: 37.79
Epoch [3/100], Val Loss: 0.7660, Accuracy: 0.8243, IOU: 0.3270, Time: 37.46
Epoch [4/100], Val Loss: 0.5264, Accuracy: 0.8003, IOU: 0.3255, Time: 37.46
Epoch [5/100], Val Loss: 0.0980, Accuracy: 0.9808, IOU: 0.4385, Time: 37.50
Epoch [6/100], Val Loss: 0.1488, Accuracy: 0.9545, IOU: 0.4204, Time: 798.72
Epoch [7/100], Val Loss: 0.3910, Accuracy: 0.9041, IOU: 0.3877, Time: 38.18
Epoch [8/100], Val Loss: 0.8785, Accuracy: 0.7814, IOU: 0.2698, Time: 37.42
Epoch [9/100], Val Loss: 1.5738, Accuracy: 0.7292, IOU: 0.2796, Time: 37.34
Epoch [10/100], Val Loss: 0.0493, Accuracy: 0.9891, IOU: 0.4493, Time: 37.27
Epoch [11/100], Val Loss: 0.3924, Accuracy: 0.8695, IOU: 0.3544, Time: 37.23
Epoch [12/100], Val Loss: 0.7538, Accuracy: 0.7422, IOU: 0.3223, Time: 37.27
Epoch [13/100], Val Loss: 0.0824, Accuracy: 0.9804, IOU: 0.4235, Time: 37.51
Epoch [

In [11]:
torch.cuda.empty_cache()

RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [1]:
%matplotlib inline
import matplotlib.pyplot as plt


def visualize_predictions_2d(model, dataloader):
    model.eval()
    for images, masks in dataloader:
      #images, masks = next(iter(dataloader))  # Get a single batch
      images, masks = images.to(device), masks.to(device)

      with torch.no_grad():
          outputs = model(images)
          preds = torch.argmax(outputs, dim=1)

      # set areas of prediction that is meant to be the background class as such:
      #preds[masks == 255] = 255

    print("Inside visualization:", preds[:5])  # First few values

    # Compute the number of identical pixels
    matching_pixels = (preds == masks).sum().item()
    total_pixels = masks.numel()
    matching_percentage = (matching_pixels / total_pixels) * 100

    print(f"Matching Pixels: {matching_pixels}/{total_pixels} ({matching_percentage:.2f}%)")


      # Display predictions
    for i in range(min(10, len(images))):  # Show up to 4 examples

          plt.figure(figsize=(12, 4))

          # Collapse hyperspectral input to 2D by averaging across bands
          input_2d = images[i].mean(dim=0).cpu()  # Mean across channels

          plt.subplot(1, 3, 1)
          plt.title("Input (2D Projection)")
          plt.imshow(input_2d)  # Projected to 2D

          plt.subplot(1, 3, 2)
          plt.title("Ground Truth")
          plt.imshow(masks[i].cpu(), cmap='tab20')

          plt.subplot(1, 3, 3)
          plt.title("Prediction")
          plt.imshow(preds[i].cpu(), cmap='tab20')

          plt.show()

# Visualize predictions
visualize_predictions_2d(model, test_loader)


NameError: name 'model' is not defined

In [8]:
#%% Inference & Visualization (batched)
from torch.utils.data import TensorDataset, DataLoader
import random
model = SimpleSegmentationModel(in_channels=input_shape[0], out_channels=nb_classes).to(device)
# Create a TensorDataset and DataLoader for the inference data
inference_tensor = torch.tensor(data, dtype=torch.float32)  # data is on CPU
inference_dataset = TensorDataset(inference_tensor)
inference_loader = DataLoader(inference_dataset, batch_size=128, shuffle=False)

all_preds = []
with torch.no_grad():
    for (batch,) in inference_loader:
        batch = batch.to(device)  # Move the current batch to GPU
        outputs = model(batch)
        preds_batch = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.append(preds_batch)

final_preds = np.concatenate(all_preds, axis=0)

# Visualize a random patch's prediction alongside its ground truth
rand_idx = random.randint(0, final_preds.shape[0] - 1)

plt.figure()
plt.imshow(final_preds[rand_idx], cmap='nipy_spectral')
plt.title("Predicted Segmentation for a Random Patch")

plt.figure()
plt.imshow(gt[rand_idx], cmap='nipy_spectral')
plt.title("Ground Truth for the Random Patch")
plt.show()


NameError: name 'input_shape' is not defined